In [1]:
from pathlib import Path

import chromadb
from chromadb.errors import NotFoundError
from llama_index.core import Settings, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore

In [7]:
PROJECT_ROOT = Path(r"C:\Program Files\Studying\coding\RAG_project")
PDF_DIR = PROJECT_ROOT / "data" / "wikipedia"
CHROMA_DIR = PROJECT_ROOT / "chromadb_store"
COLLECTION_NAME = "mini_wikipedia"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"

if not PDF_DIR.exists():
    raise FileNotFoundError(f"PDF folder not found: {PDF_DIR}")

# LlamaIndex uses Settings as global defaults for later index and query operations.
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)
Settings.llm = None

print(f"PDF source folder: {PDF_DIR}")
print(f"ChromaDB folder: {CHROMA_DIR}")
print(f"Embedding model: {EMBED_MODEL_NAME}")

LLM is explicitly disabled. Using MockLLM.
PDF source folder: C:\Program Files\Studying\coding\RAG_project\data\wikipedia
ChromaDB folder: C:\Program Files\Studying\coding\RAG_project\chromadb_store
Embedding model: BAAI/bge-small-en-v1.5


In [8]:
def get_chroma_collection(
    chroma_dir: Path,
    collection_name: str,
    reset_collection: bool = False,
):
    """Create or load a persistent ChromaDB collection."""
    # PersistentClient writes vectors and metadata to disk instead of keeping them in memory.
    chroma_dir.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(chroma_dir))

    # Some ChromaDB versions raise NotFoundError when the collection is missing,
    # while older versions may raise ValueError. Catch both so reset stays safe.
    if reset_collection:
        try:
            client.delete_collection(collection_name)
            print(f"Deleted existing collection: {collection_name}")
        except (NotFoundError, ValueError):
            print(f"Collection did not exist yet: {collection_name}")

    return client.get_or_create_collection(collection_name)


In [9]:
collection = get_chroma_collection(CHROMA_DIR, COLLECTION_NAME)

In [10]:
def load_existing_index(
    chroma_dir: Path = CHROMA_DIR,
    collection_name: str = COLLECTION_NAME,
) -> VectorStoreIndex:
    """Reconnect to an existing persisted ChromaDB collection without reinserting data."""
    # This is the path you use after closing and reopening the notebook.
    collection = get_chroma_collection(chroma_dir, collection_name, reset_collection=False)
    vector_store = ChromaVectorStore(chroma_collection=collection)
    return VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=Settings.embed_model)


def retrieve_relevant_chunks(
    query: str,
    top_k: int = 5,
    query_index: VectorStoreIndex | None = None,
):
    """Retrieve the most relevant chunks for a user question."""
    # Use the provided index when one is passed in, otherwise fall back to the current session index.
    active_index = query_index if query_index is not None else index
    retriever = active_index.as_retriever(similarity_top_k=top_k)
    results = retriever.retrieve(query)

    # Print source metadata beside each chunk so you can inspect where answers come from.
    for rank, result in enumerate(results, start=1):
        metadata = result.node.metadata
        file_name = metadata.get("file_name", "unknown file")
        page_label = metadata.get("page_label", metadata.get("page", "unknown page"))
        chunk_number = metadata.get("chunk_number", "unknown chunk")

        print(f"\n--- Result {rank} | score={result.score:.4f} ---")
        print(f"Source: {file_name} | page: {page_label} | chunk: {chunk_number}")
        print(result.node.get_content()[:1200])

    return results

In [12]:
# If you reopen the notebook later, rerun the setup cells and then use this block.
reloaded_index = load_existing_index()
sample_query = 'I am Jason. Do you think what vector are the best for me ?'
retrieved_chunks_after_reopen = retrieve_relevant_chunks(sample_query, top_k=3, query_index=reloaded_index)


--- Result 1 | score=0.4109 ---
Source: Reinforcement learning.pdf | page: 65 | chunk: 673
ρ
 (
 θ
 )
 =
 
 ρ
 
 
 π
 
 θ
 
 
 
 
 
 
 {\displaystyle \rho (\theta )=\rho ^{\pi _{\theta }}}
 
 under mild conditions this function will be differentiable as a function of the parameter vector 
 
 
 
 θ

--- Result 2 | score=0.4064 ---
Source: Reinforcement learning.pdf | page: 66 | chunk: 674
{\displaystyle \theta }
 
. If the gradient of 
 
 
 
 ρ
 
 
 {\displaystyle \rho }
 
 was known, one could use gradient ascent. Since an analytic expression for the gradient is not
available, only a noisy estimate is available. Such an estimate can be constructed in many ways,
giving rise to algorithms such as Williams's REINFORCE method (which is known as the likelihood
ratio method in the simulation-based optimization literature).
A large class of methods avoids relying on gradient information. These include simulated annealing,
cross-entropy search or methods of evolutionary computation. Many grad